In [1]:
from typing import Any, Optional

import torch
import numpy as np
from torch import Tensor
import plotly.express as px
import matplotlib.pyplot as plt
from torchvision.transforms import v2
from torch.utils.data import WeightedRandomSampler, TensorDataset, DataLoader

from src import configs as cfg
from src import dataset, plotting

In [13]:
x_train, y_train, x_test = dataset.load_raw_dataset(cfg.DEVICE)
x_train, y_train = dataset.remove_samples_without_labels(x_train, y_train)

In [25]:
def cls_presence_mask(y_train: Tensor) -> Tensor:
    cls_presence = torch.empty(y_train.shape[0], cfg.N_CLASSES)
    for class_idx in range(0, cfg.N_CLASSES):
        cls_presence[:, class_idx] = (y_train == class_idx).any(dim=(1, 2))
    return cls_presence
y_classes = cls_presence_mask(y_train)

In [15]:
class_counts = y_classes.sum(dim=0, keepdim=True)
cls_weight = 1 / (class_counts + 1e-8)
cls_weight

tensor([[0.0013, 0.0476, 0.0556, 0.0044, 0.0032, 0.0032, 0.0667, 0.0588, 0.0769,
         0.0058, 0.0189, 0.0077, 0.0270, 0.0833, 0.1111, 0.0323, 0.0161, 0.0149,
         0.0135, 0.0122, 0.0294, 0.0244, 0.0143, 0.0102, 0.0119, 0.1667, 0.1667,
         0.0127, 0.0110, 0.0112, 0.0122, 0.0064, 0.0060, 0.0067, 0.0147, 0.0137,
         0.0080, 0.0069, 0.0065, 0.0169, 0.0263, 0.0286, 0.0051, 0.0052, 0.0143,
         0.0161, 0.0161, 0.0065, 0.0161, 0.0130, 0.0179, 0.0556, 0.0286, 0.0085,
         0.0060]])

In [16]:
normed_cls_weight = cls_weight / cls_weight.sum()

In [17]:
sample_weights = (y_classes * cls_weight).sum(dim=1)

In [57]:
train_ds = TensorDataset(x_train, y_train)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

B_SIZE = 10

unweighted_data_loader = DataLoader(train_ds, B_SIZE, shuffle=True, drop_last=True)
weighted_data_loader = DataLoader(train_ds, B_SIZE, sampler=sampler, drop_last=True)

In [63]:
unweighted_x, unweighted_y_true = next(iter(unweighted_data_loader))
unweighted_batch = {"x": unweighted_x, "y_true": unweighted_y_true}
weighted_x, weighted_y_true = next(iter(weighted_data_loader))
weighted_batch = {"x": weighted_x, "y_true": weighted_y_true}
def n_unique_classes_in_batch(y_train: Tensor) -> int:
    cls_presence = cls_presence_mask(y_train)
    cls_present_in_batch = cls_presence.any(dim=0)
    n_cls_in_batch = cls_present_in_batch.sum()
    # print("n classes in batch:", n_cls_in_batch)
    return n_cls_in_batch

diffs = []
for (_, unweighted_y), (_, weighted_y) in zip(unweighted_data_loader, weighted_data_loader):
    n_cls_in_unweighted = n_unique_classes_in_batch(unweighted_y)
    prev_batch = n_cls_in_unweighted
    n_cls_in_weighted = n_unique_classes_in_batch(weighted_y)
    diff = n_cls_in_weighted - n_cls_in_unweighted
    diffs.append(diff)
    # print("n cls diff:", diff)
    # print()

print("mean diff:", torch.stack(diffs).float().mean().item())


mean diff: 4.493333339691162


In [19]:
@torch.no_grad
def plt_batch(
        batch: dict[str, Any],
        transform: Optional[v2.Transform] = v2.Identity(),
        ax_size: Optional[int] = 4,
    ):
    batch = dataset.preprocess_batch(batch)
    n_samples_to_plt = len(batch["x"])
    n_rows = 2
    batch["x"] = batch["x"].detach().cpu().numpy().squeeze()
    batch["y_true"] = batch["y_true"].detach().cpu().numpy()
    fig, axes = plt.subplots(n_rows, n_samples_to_plt, squeeze=False)
    fig.set_size_inches(n_samples_to_plt * ax_size, n_rows * ax_size)
    for j in range(n_samples_to_plt):
        axes[0, j].imshow(
            batch["x"][j],
            cmap="gray",
        )
        imshow_seg(axes[1, j], batch["x"][j], batch["y_true"][j])

def imshow_seg(ax, x: Tensor, seg: Tensor):
    ax.imshow(x, cmap="gray")
    seg_masked = np.ma.masked_where(seg == 0, seg)
    ax.imshow(seg_masked, cmap="tab20")